# NB11：PLS 迴歸定量 I — 醋酸（Acetic Acid）NIR 定量校正

**食品分析實驗 週次 13｜對應 Nielsen's Food Analysis, 6th ed. 光譜法／化學計量學章節，銜接 W13 課堂 deck**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 用共用的長格式光譜資料結構（`load_spectra` / `split_spectra` / `plot_spectra`）載入並檢視 NIR 光譜。
2. 用 **Kennard–Stone 演算法** 把樣品切成校正集（calibration set）與測試集（test set），並解釋為什麼測試集絕不能拿來調參。
3. 比較不同前處理方法（不做前處理 / SNV / Savitzky–Golay 一階導數）對 **RMSECV** 的影響。
4. 畫出 **RMSEC、RMSECV vs LV 數（1–15）** 的曲線，判斷欠擬合／過擬合，並選出最佳潛在變數（latent variable, LV）數。
5. 用**分組交叉驗證**（`GroupKFold` / `LeaveOneGroupOut`，以 `sample_id` 分組）避免同一樣品的重複測量同時出現在訓練與驗證折中造成的資料外洩（data leakage）。
6. 計算最終模型的 **RMSEP、R²、bias、RPD**，並整理成一張 model report card。
7. 看懂等效的 R（`pls` 套件）與 Orange（Test and Score）建模流程。

## 如何使用本筆記本

- 直接「全部執行」（Run all）即可用內建的**模擬光譜資料**跑完整份筆記本。
- 若要換成你們班上 InnoSpectra／Avantes 實際量測的光譜，把資料整理成本筆記本使用的共用長格式（見下方「資料格式」），存成 CSV 後傳入 `load_spectra("你的檔案路徑或URL")`；本筆記本第 9 節示範了這個用法。
- 若你的資料是用 **SpectraView** 匯出（File → Export → Combined CSV，`layout="rows"`），`load_spectra()` 會自動偵測並轉換格式，不需要手動改欄位。

> ⚠️ **本筆記本內建的資料是模擬教學資料（模擬光譜）**，不是真實量測值，僅用於練習 PLS 建模流程與寫法。做作業時請換成你們班的實際數據。

本筆記本使用的資料欄位重點：`sample_id`（樣品唯一代號，**同一樣品的所有重複測量必須共用同一個 `sample_id`**，交叉驗證要用它分組）、`group`（本筆記本內建資料固定為 `0`，代表教師提供的示範資料）、`instrument`、`rep`、以及目標值欄位 `acidity_pct`（醋酸濃度 % w/v，本週的參考值來自 W4 手動滴定）。完整的共用光譜資料格式規範見下方 Setup 小節。

## 1. 環境設定 (Setup)

匯入套件，並定義本課程所有光譜筆記本共用的三個函式：`load_spectra()`、`split_spectra()`、`plot_spectra()`。

**注意**：圖表座標軸與圖例一律使用英文，避免 Colab 預設字型無法顯示中文而出現方框亂碼；文字說明使用繁體中文。

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import savgol_filter
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut, KFold

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 13
np.random.seed(RNG_SEED)

PREP_SD = 0.05  # %w/v, W4 titration reference-value noise baked into the demo data

### 光譜資料格式 (Shared Spectral Data Format) — NB09–NB13 共用

從本週開始，資料改用**寬格式 (wide format)**，跟 NB01 的長格式不同：**一列＝一條完整光譜**。

| 規則 | 說明 |
|---|---|
| 中繼資料欄位在前 | 欄名**不是數字**的欄位，例如 `sample_id`（樣品代號）、`group`（組別 1–5，沿用 NB01 的定義；TA 提供的參考/示範資料用 `0`）、`instrument`（儀器代號）、`rep`（重複次數）、`role`（`library`/`unknown`/`mixture` 等角色）、以及依主題而定的標籤欄位，例如 `class`（化合物名稱）、`oil`（油品代號）、`acidity_pct`、`aspartame_pct`、`sucralose_pct` 等。 |
| 光譜欄位在後 | 欄名**能轉成浮點數**的欄位＝光譜資料本身，欄名即該點的波長 (nm，NIR) 或波數 (cm⁻¹，Raman)。程式一律依欄名數值排序，不假設檔案裡欄位順序已經由小到大。 |
| 光譜欄位不可缺值 | 每一列在所有光譜欄位都必須有數值，不可有 NaN。 |
| **一個檔案只用一種共同 x 軸** | Raman 位移與 NIR 波長不可混在同一個檔案；不同解析度/範圍的儀器資料也請分開存檔，或先內插到同一組 x 軸再合併。 |
| 標籤欄位的值用英文代碼 | 例如油品用 `EVOO`/`COC`，化合物用 `aspartame`/`sucralose`。畫圖的圖例文字直接取自欄位值，中文會因 Colab 字型問題變成方框。 |

**也相容 SpectraView 的「rows」匯出格式**（`specview/formats/__init__.py` 的 `save_combined_csv(..., layout="rows")`）：
第一列表頭的第一格是 `x:<軸標籤>`（例如 `x:Raman shift (cm-1)`），之後每一格才是 x 值本身；
從第二列開始，每一列的第一格是**光譜名稱**、其餘是強度值。這種格式沒有額外中繼資料欄位，
只有一個 `sample_id`——如果要加上組別/儀器等資訊，請自行用 `sample_id` 做 `merge`。

共用讀取/拆解/畫圖函式（簽名固定，NB09–NB13 都一樣）：

```python
load_spectra(source=None)          # -> 寬格式 DataFrame；None 時退回內建模擬光譜
split_spectra(df)                  # -> (meta, X, axis)
plot_spectra(axis, X, labels)      # -> matplotlib Axes，依 labels 分色、圖例自動去重
```

In [ ]:
def _is_number(text):
    try:
        float(text)
        return True
    except (TypeError, ValueError):
        return False


def load_spectra(source=None):
    """讀取寬格式光譜 CSV（見上方「光譜資料格式」說明）。

    Parameters
    ----------
    source : str, path, file-like, or None
        - None：退回本筆記本內嵌的模擬光譜字串 SAMPLE_CSV（模擬光譜，非真實量測）。
        - 本機/雲端硬碟路徑，或 http(s) URL（例如已發布的 Google 試算表 CSV 連結）：
          直接用 pandas.read_csv 讀取。

    Returns
    -------
    pandas.DataFrame
        寬格式：中繼資料欄位（欄名非數字）+ 光譜欄位（欄名為數字，代表 nm 或 cm-1）。
    """
    if source is None:
        return pd.read_csv(io.StringIO(SAMPLE_CSV))
    return pd.read_csv(source)


def split_spectra(df):
    """把 load_spectra() 讀進來的寬格式 DataFrame 拆成 (meta, X, axis)。

    自動判斷兩種輸入：
    - 一般格式：欄名可轉成浮點數的欄位＝光譜欄位；其餘欄位＝中繼資料 (meta)。
    - SpectraView「rows」匯出格式：第一個欄名以 "x:" 開頭，代表該欄其實是
      「樣品名稱」，其餘欄名本身就是 x 值；此時 meta 只有一欄 sample_id
      （由第一欄改名而來），其他中繼資料需自行用 sample_id 合併(merge)。

    Returns
    -------
    meta : pandas.DataFrame   中繼資料（含 sample_id 或其他非光譜欄位）
    X    : numpy.ndarray, shape (n_spectra, n_points)   光譜強度矩陣
    axis : numpy.ndarray, shape (n_points,)             x 軸，已由小到大排序
    """
    cols = list(df.columns)

    if isinstance(cols[0], str) and cols[0].startswith("x:"):
        meta = df.iloc[:, [0]].rename(columns={cols[0]: "sample_id"}).reset_index(drop=True)
        spec_cols = cols[1:]
    else:
        spec_cols = [c for c in cols if _is_number(c)]
        meta_cols = [c for c in cols if c not in spec_cols]
        meta = df[meta_cols].reset_index(drop=True)

    axis = np.array([float(c) for c in spec_cols])
    X = df[spec_cols].to_numpy(dtype=float)

    order = np.argsort(axis)
    axis = axis[order]
    X = X[:, order]
    return meta, X, axis


def plot_spectra(axis, X, labels, ax=None, title=None, xlabel="x", ylabel="Intensity (a.u.)"):
    """畫出多條光譜，依 labels 分色，圖例只顯示每個 label 一次。"""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    labels = list(labels)
    unique_labels = sorted(set(labels), key=labels.index)
    cmap = plt.get_cmap("tab10")
    color_of = {lab: cmap(i % 10) for i, lab in enumerate(unique_labels)}
    seen = set()
    for i in range(X.shape[0]):
        lab = labels[i]
        show_label = lab if lab not in seen else None
        seen.add(lab)
        ax.plot(axis, X[i], color=color_of[lab], linewidth=1.3, alpha=0.85, label=show_label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    return ax

In [ ]:
# ---- Embedded demo spectra (fallback) ---------------------------------
# Exact content of data/nb11_demo_spectra.csv, embedded so the notebook
# is self-contained even without the data/ folder present.
SAMPLE_CSV = """sample_id,group,instrument,rep,acidity_pct,900,913.56,927.12,940.68,954.24,967.8,981.36,994.92,1008.47,1022.03,1035.59,1049.15,1062.71,1076.27,1089.83,1103.39,1116.95,1130.51,1144.07,1157.63,1171.19,1184.75,1198.31,1211.86,1225.42,1238.98,1252.54,1266.1,1279.66,1293.22,1306.78,1320.34,1333.9,1347.46,1361.02,1374.58,1388.14,1401.69,1415.25,1428.81,1442.37,1455.93,1469.49,1483.05,1496.61,1510.17,1523.73,1537.29,1550.85,1564.41,1577.97,1591.53,1605.08,1618.64,1632.2,1645.76,1659.32,1672.88,1686.44,1700
STD-00,0,InnoSpectra_NIR-S_demo,1,0.0,0.078514,0.103975,0.092355,0.121097,0.094808,0.089727,0.112605,0.114317,0.105137,0.111233,0.131902,0.13588,0.119744,0.126127,0.105908,0.169439,0.141412,0.172428,0.161498,0.192475,0.190882,0.175585,0.200545,0.163922,0.205722,0.163437,0.199655,0.219351,0.216369,0.223781,0.242863,0.289925,0.369243,0.44164,0.51771,0.614765,0.772529,0.862158,0.991328,1.075763,1.151739,1.149989,1.116161,1.036785,0.917206,0.820907,0.718811,0.587836,0.510343,0.428675,0.399025,0.36029,0.34426,0.324984,0.342802,0.308462,0.310812,0.309526,0.333683,0.316722
STD-00,0,InnoSpectra_NIR-S_demo,2,0.0,0.219904,0.213943,0.162598,0.19603,0.198024,0.199776,0.217403,0.259265,0.227169,0.223923,0.252337,0.245664,0.250572,0.240881,0.25657,0.268049,0.232299,0.241683,0.291998,0.292715,0.274891,0.271381,0.290564,0.277925,0.296484,0.33441,0.313599,0.338405,0.32591,0.353226,0.396838,0.389559,0.461781,0.53994,0.642868,0.778064,0.888226,0.976644,1.110841,1.187955,1.238169,1.294886,1.204521,1.150252,1.030448,0.940113,0.814281,0.703894,0.61134,0.563016,0.478357,0.449052,0.422638,0.453795,0.421774,0.408605,0.420806,0.473315,0.4325,0.45258
STD-00,0,InnoSpectra_NIR-S_demo,3,0.0,0.260875,0.270347,0.273383,0.24523,0.330199,0.305268,0.294902,0.292847,0.286257,0.319204,0.303263,0.317966,0.289611,0.323014,0.353974,0.318459,0.34046,0.331075,0.353114,0.356643,0.344013,0.33802,0.35675,0.356433,0.348614,0.370302,0.374252,0.358361,0.382044,0.450198,0.440105,0.46622,0.562013,0.585849,0.6463,0.768913,0.899768,1.054901,1.17977,1.213325,1.319205,1.285359,1.272066,1.198583,1.093767,0.968764,0.853682,0.724205,0.658567,0.60088,0.54984,0.545075,0.49863,0.481399,0.492335,0.466979,0.441988,0.428078,0.490711,0.481816
STD-01,0,InnoSpectra_NIR-S_demo,1,0.7692,0.257057,0.234443,0.21348,0.228804,0.263379,0.224997,0.203336,0.211917,0.231843,0.271889,0.263497,0.252814,0.24754,0.271167,0.274179,0.271307,0.28129,0.278263,0.329773,0.269579,0.325165,0.319841,0.360681,0.350927,0.361247,0.359961,0.349886,0.362145,0.32688,0.38044,0.384998,0.433413,0.509286,0.579395,0.694555,0.742007,0.891589,1.046544,1.116908,1.213402,1.291509,1.281922,1.212195,1.15992,1.059813,0.933218,0.813517,0.711392,0.594584,0.553367,0.485271,0.475488,0.456105,0.422865,0.443826,0.438625,0.433225,0.44884,0.434356,0.438309
STD-01,0,InnoSpectra_NIR-S_demo,2,0.7692,0.27469,0.218254,0.256671,0.231033,0.259679,0.27591,0.274924,0.294607,0.286846,0.238104,0.268957,0.31772,0.321102,0.299583,0.274893,0.341789,0.273377,0.319654,0.321023,0.336295,0.325091,0.351797,0.388089,0.367322,0.35185,0.370592,0.367461,0.362149,0.355866,0.391887,0.404629,0.424616,0.503901,0.576758,0.67994,0.767536,0.910653,1.053536,1.16729,1.237399,1.279892,1.293845,1.25945,1.196161,1.070466,0.93576,0.836427,0.688562,0.63356,0.555869,0.513243,0.488781,0.460994,0.424357,0.437994,0.429282,0.443617,0.452332,0.432639,0.464647
STD-01,0,InnoSpectra_NIR-S_demo,3,0.7692,0.331594,0.297151,0.316309,0.345043,0.301786,0.326339,0.322858,0.360783,0.363873,0.336928,0.352778,0.309397,0.371429,0.366094,0.385937,0.326229,0.376758,0.370069,0.338996,0.38177,0.4049,0.433371,0.434081,0.427672,0.443568,0.429509,0.415215,0.407275,0.431844,0.445156,0.497709,0.496401,0.586694,0.642294,0.756363,0.839826,0.967388,1.09878,1.224689,1.292284,1.347523,1.344157,1.317202,1.226189,1.109978,1.010346,0.885742,0.807143,0.685263,0.620406,0.563285,0.554688,0.529364,0.514248,0.476804,0.470833,0.501932,0.511715,0.527208,0.474452
STD-02,0,InnoSpectra_NIR-S_demo,1,1.5385,0.190078,0.212751,0.189971,0.189257,0.198572,0.227933,0.250166,0.185425,0.209531,0.209409,0.239137,0.229219,0.250558,0.209747,0.229906,0.225052,0.218343,0.240766,0.233752,0.306568,0.297316,0.294882,0.319201,0.36615,0.323915,0.330773,0.314036,0.312475,0.303249,0.298281,0.313574,0.330081,0.414991,0.49733,0.593652,0.721804,0.833011,1.00885,1.061236,1.124408,1.172351,1.181309,1.15685,1.025819,0.971939,0.835475,0.725597,0.639186,0.545415,0.451245,0.410847,0.410432,0.380744,0.360541,0.340201,0.354463,0.34508,0.33042,0.324763,0.35299
STD-02,0,InnoSpectra_NIR-S_demo,2,1.5385,0.228909,0.265426,0.239829,0.271852,0.264246,0.249604,0.247766,0.266277,0.268289,0.293222,0.269938,0.261024,0.290301,0.32781,0.327608,0.294223,0.306233,0.3247,0.320352,0.345119,0.367107,0.377792,0.404385,0.415759,0.425259,0.389995,0.398953,0.372164,0.375581,0.381533,0.492536,0.484256,0.543203,0.615243,0.704139,0.835673,0.970756,1.09365,1.195883,1.285897,1.252025,1.296495,1.279426,1.186581,1.088007,1.000707,0.863228,0.785435,0.66679,0.603876,0.533268,0.485676,0.493259,0.491504,0.480758,0.463665,0.499897,0.493272,0.496159,0.486994
STD-02,0,InnoSpectra_NIR-S_demo,3,1.5385,0.296015,0.293502,0.304465,0.333702,0.357333,0.31445,0.311681,0.292816,0.375891,0.314234,0.352553,0.350915,0.369801,0.337214,0.329155,0.364531,0.375325,0.396503,0.360873,0.399716,0.425698,0.431245,0.47099,0.49671,0.480025,0.445204,0.462874,0.463521,0.458752,0.474607,0.45698,0.531734,0.591787,0.656501,0.768918,0.88334,1.049045,1.143389,1.232368,1.326014,1.367382,1.380681,1.317325,1.245004,1.161283,1.077728,0.932479,0.796739,0.735446,0.651863,0.585408,0.573567,0.55657,0.563798,0.575572,0.572666,0.569439,0.551874,0.56607,0.5672
STD-03,0,InnoSpectra_NIR-S_demo,1,2.3077,0.258565,0.245833,0.23127,0.244636,0.247284,0.250864,0.264186,0.259713,0.253246,0.250665,0.232785,0.257667,0.287516,0.274873,0.263073,0.273861,0.268874,0.297423,0.341834,0.361869,0.382474,0.408414,0.417115,0.409867,0.422664,0.3929,0.389376,0.358675,0.338573,0.362085,0.424445,0.395781,0.487416,0.561141,0.660903,0.781805,0.928351,1.047014,1.146277,1.209562,1.238759,1.221661,1.198523,1.10904,0.99637,0.913356,0.770669,0.655977,0.589253,0.4862,0.468304,0.443802,0.408532,0.408358,0.435661,0.421658,0.413884,0.410007,0.39338,0.402477
STD-03,0,InnoSpectra_NIR-S_demo,2,2.3077,0.2378,0.225449,0.270074,0.228323,0.255626,0.270456,0.255781,0.248295,0.263637,0.299517,0.280552,0.268407,0.286711,0.276917,0.252045,0.292424,0.311976,0.32341,0.318346,0.319375,0.395247,0.405779,0.447332,0.464548,0.44305,0.414805,0.380327,0.364892,0.365687,0.363792,0.398684,0.396227,0.469602,0.569699,0.67046,0.805761,0.92933,1.075879,1.182597,1.202614,1.275532,1.231697,1.219406,1.107074,1.047064,0.923625,0.810039,0.723211,0.624575,0.545782,0.483168,0.46522,0.432594,0.426323,0.453839,0.442535,0.433636,0.441718,0.409383,0.438907
STD-03,0,InnoSpectra_NIR-S_demo,3,2.3077,0.191149,0.194025,0.202275,0.195731,0.196682,0.198512,0.225196,0.226157,0.182037,0.211734,0.234224,0.210674,0.22151,0.21034,0.203147,0.229439,0.225465,0.238464,0.231561,0.259656,0.307825,0.335324,0.377855,0.360412,0.38159,0.357066,0.323425,0.313353,0.293181,0.297083,0.320044,0.335298,0.412802,0.509731,0.580282,0.751183,0.861863,0.981694,1.079008,1.145945,1.200484,1.174696,1.166195,1.061314,0.965914,0.85299,0.714014,0.600623,0.532909,0.474061,0.427157,0.373152,0.388606,0.345243,0.330544,0.360231,0.361966,0.3269,0.359533,0.326715
STD-04,0,InnoSpectra_NIR-S_demo,1,3.0769,0.335619,0.339279,0.318905,0.343174,0.349722,0.329127,0.354704,0.365104,0.364012,0.372212,0.3656,0.354951,0.379118,0.350565,0.375917,0.412294,0.431897,0.420918,0.393485,0.488157,0.515466,0.546521,0.596648,0.606858,0.566905,0.537196,0.54421,0.462241,0.499993,0.499699,0.531424,0.582077,0.634425,0.725461,0.831433,0.965859,1.113256,1.208992,1.27839,1.351753,1.359072,1.376726,1.382496,1.281795,1.169333,1.048337,0.914598,0.832071,0.741586,0.687964,0.641568,0.607583,0.557525,0.604869,0.584009,0.57668,0.573217,0.58955,0.607183,0.600587
STD-04,0,InnoSpectra_NIR-S_demo,2,3.0769,0.188537,0.233015,0.24526,0.256063,0.239908,0.253769,0.22663,0.25329,0.24461,0.237697,0.222756,0.273198,0.217632,0.268923,0.247331,0.27476,0.260653,0.293059,0.294755,0.309551,0.357104,0.379783,0.443828,0.468317,0.445122,0.409687,0.394953,0.325642,0.341345,0.334399,0.344497,0.397331,0.425571,0.52787,0.652107,0.764131,0.915546,1.040364,1.130103,1.154942,1.217204,1.194582,1.160402,1.072867,0.992621,0.851416,0.752188,0.648216,0.566034,0.466605,0.447519,0.401312,0.404499,0.3621,0.352435,0.358994,0.354683,0.358316,0.341441,0.329707
STD-04,0,InnoSpectra_NIR-S_demo,3,3.0769,0.201921,0.19219,0.18403,0.200557,0.207121,0.255867,0.233529,0.217846,0.19279,0.217389,0.218063,0.235546,0.26507,0.230836,0.201667,0.237216,0.216857,0.270383,0.293655,0.309985,0.318177,0.382243,0.438444,0.412767,0.407682,0.392519,0.362332,0.328195,0.319613,0.320244,0.325641,0.370151,0.433951,0.496236,0.634859,0.771312,0.91364,1.024136,1.083428,1.137448,1.194587,1.185994,1.150028,1.06265,0.948207,0.859066,0.736792,0.644405,0.564479,0.496112,0.444973,0.378785,0.338815,0.352897,0.337452,0.333998,0.368114,0.331338,0.376563,0.3823
STD-05,0,InnoSpectra_NIR-S_demo,1,3.8462,0.234139,0.23868,0.27212,0.238392,0.285342,0.266536,0.243027,0.313097,0.267856,0.334559,0.272269,0.288204,0.311473,0.313417,0.292027,0.327449,0.338292,0.358291,0.349009,0.405593,0.466212,0.537323,0.570039,0.578322,0.562763,0.557749,0.475576,0.459871,0.400346,0.467352,0.46549,0.465647,0.566784,0.671717,0.787947,0.923563,1.078909,1.16309,1.248004,1.297438,1.333226,1.312586,1.315504,1.216689,1.101292,1.003751,0.901429,0.79409,0.727022,0.636766,0.583126,0.574299,0.56967,0.522645,0.50921,0.526051,0.53866,0.526573,0.551542,0.539108
STD-05,0,InnoSpectra_NIR-S_demo,2,3.8462,0.307819,0.326883,0.339468,0.345981,0.320935,0.335288,0.346462,0.363862,0.326803,0.323097,0.354821,0.34467,0.374209,0.354175,0.362055,0.347038,0.371739,0.400331,0.398182,0.418787,0.503328,0.535754,0.582643,0.583647,0.594656,0.526045,0.48415,0.454893,0.43085,0.394497,0.456954,0.464098,0.546346,0.62407,0.751618,0.886555,1.020349,1.149134,1.224063,1.285841,1.303365,1.277791,1.268421,1.209879,1.0825,0.952001,0.823667,0.736809,0.622996,0.592512,0.523038,0.486391,0.47979,0.442559,0.428292,0.448346,0.476519,0.442643,0.473685,0.429759
STD-05,0,InnoSpectra_NIR-S_demo,3,3.8462,0.26281,0.206354,0.238973,0.228479,0.252883,0.24212,0.257051,0.246287,0.253519,0.273532,0.295877,0.277916,0.280274,0.274268,0.313188,0.258785,0.266007,0.300606,0.345652,0.392381,0.436673,0.493996,0.526179,0.531013,0.50533,0.492462,0.423912,0.411003,0.397445,0.374229,0.394932,0.429642,0.510077,0.591342,0.701877,0.86454,1.026372,1.115384,1.18793,1.228804,1.246833,1.259615,1.221029,1.139976,1.064613,0.927071,0.814675,0.717326,0.63664,0.530461,0.517172,0.502047,0.442789,0.457917,0.438457,0.429715,0.452458,0.449,0.418908,0.447683
STD-06,0,InnoSpectra_NIR-S_demo,1,4.6154,0.233914,0.235368,0.25812,0.23397,0.231336,0.261016,0.273204,0.27099,0.27605,0.253684,0.242534,0.25824,0.27245,0.247096,0.273858,0.293236,0.279519,0.339274,0.345194,0.354946,0.413963,0.482774,0.554157,0.566519,0.529101,0.494495,0.450477,0.401142,0.340017,0.344481,0.362654,0.383901,0.486607,0.553145,0.681433,0.829907,0.9916,1.102022,1.171328,1.222926,1.208274,1.24552,1.218872,1.092774,0.994497,0.896688,0.793301,0.664106,0.568047,0.54906,0.456751,0.386216,0.421257,0.417748,0.424418,0.423835,0.446617,0.386628,0.381641,0.376175
STD-06,0,InnoSpectra_NIR-S_demo,2,4.6154,0.314183,0.310755,0.350757,0.344322,0.359927,0.343084,0.348133,0.371556,0.363196,0.379395,0.370438,0.370769,0.384767,0.376241,0.388319,0.397658,0.38942,0.411031,0.441768,0.494547,0.545049,0.60715,0.658681,0.662282,0.653725,0.58861,0.515872,0.509271,0.44417,0.470709,0.514832,0.536334,0.566345,0.688111,0.808457,0.942113,1.097405,1.196306,1.267825,1.321144,1.311425,1.346455,1.325428,1.213587,1.098974,0.994792,0.908601,0.775855,0.705704,0.66893,0.569596,0.551069,0.519779,0.512174,0.515382,0.499392,0.520437,0.517646,0.536887,0.531553
STD-06,0,InnoSpectra_NIR-S_demo,3,4.6154,0.226271,0.244978,0.228177,0.267981,0.236291,0.269787,0.245955,0.266589,0.249719,0.259498,0.261193,0.289389,0.295678,0.304457,0.303741,0.300064,0.324598,0.356329,0.352493,0.391178,0.472947,0.56163,0.575347,0.589717,0.560092,0.520028,0.463758,0.420673,0.431521,0.432898,0.430476,0.467507,0.540838,0.621194,0.753243,0.915051,1.048841,1.179255,1.281783,1.271184,1.299789,1.27728,1.244695,1.180298,1.086787,0.964469,0.835402,0.760283,0.659754,0.610004,0.552565,0.523351,0.505973,0.462139,0.515536,0.479573,0.503491,0.510361,0.495675,0.486753
STD-07,0,InnoSpectra_NIR-S_demo,1,5.3846,0.219432,0.207044,0.189506,0.210441,0.233726,0.214967,0.246914,0.209541,0.222568,0.212554,0.229443,0.274522,0.261922,0.251257,0.249186,0.284581,0.272303,0.274889,0.349208,0.350921,0.443645,0.52335,0.584403,0.620794,0.578343,0.497834,0.465357,0.398522,0.342239,0.337672,0.356512,0.368826,0.473405,0.571167,0.704857,0.841395,1.000887,1.125593,1.172528,1.199067,1.20689,1.211778,1.193493,1.097899,1.036545,0.926187,0.784611,0.64249,0.579982,0.539505,0.46063,0.4266,0.419281,0.453868,0.395784,0.369838,0.408913,0.41313,0.421915,0.413771
STD-07,0,InnoSpectra_NIR-S_demo,2,5.3846,0.295529,0.281775,0.293955,0.275635,0.314779,0.298294,0.289833,0.300875,0.284388,0.295843,0.323066,0.322341,0.329299,0.330266,0.317103,0.328618,0.367777,0.350721,0.357408,0.432927,0.492917,0.56976,0.626997,0.628078,0.628971,0.563988,0.512663,0.43785,0.401911,0.403966,0.453234,0.429197,0.538726,0.61438,0.769269,0.901432,1.060616,1.163389,1.253529,1.250315,1.273921,1.249519,1.226516,1.13395,1.044408,0.939808,0.829721,0.701987,0.616171,0.584138,0.507169,0.486912,0.459229,0.426388,0.446287,0.44465,0.47747,0.417131,0.442894,0.465082
STD-07,0,InnoSpectra_NIR-S_demo,3,5.3846,0.288598,0.282388,0.266615,0.265551,0.272245,0.28852,0.283683,0.276979,0.295346,0.308665,0.282226,0.320285,0.295827,0.307025,0.306026,0.310846,0.313629,0.286772,0.33829,0.417366,0.450168,0.53974,0.565151,0.598459,0.609511,0.531173,0.446728,0.39612,0.34575,0.373141,0.373585,0.423172,0.482939,0.558084,0.67989,0.832422,0.990385,1.09906,1.180748,1.17479,1.214444,1.180229,1.145491,1.091925,0.985802,0.85351,0.736899,0.662075,0.552692,0.482116,0.419686,0.446056,0.413533,0.378851,0.401177,0.369098,0.354359,0.369757,0.385615,0.344111
STD-08,0,InnoSpectra_NIR-S_demo,1,6.1538,0.272905,0.288258,0.24491,0.277874,0.300975,0.297597,0.297538,0.276585,0.299468,0.277724,0.307546,0.321934,0.312244,0.333992,0.331327,0.353349,0.343628,0.343005,0.385496,0.452167,0.54295,0.62792,0.678217,0.71557,0.676554,0.636702,0.530731,0.513025,0.462792,0.442165,0.444598,0.508209,0.551667,0.6728,0.835066,0.973907,1.126093,1.232978,1.269108,1.340053,1.318143,1.325991,1.28043,1.192197,1.127377,1.000184,0.872287,0.771962,0.666586,0.61636,0.602437,0.572323,0.524345,0.541509,0.544201,0.541162,0.551657,0.501937,0.541658,0.526546
STD-08,0,InnoSpectra_NIR-S_demo,2,6.1538,0.309873,0.325835,0.287255,0.305668,0.307628,0.321672,0.298501,0.341115,0.363203,0.32866,0.344935,0.29401,0.321438,0.355894,0.353159,0.364646,0.38541,0.383231,0.412354,0.464982,0.550678,0.594518,0.718051,0.677283,0.659291,0.623554,0.563901,0.495456,0.474377,0.430547,0.439127,0.493558,0.526213,0.646946,0.792441,0.953859,1.063462,1.193895,1.215642,1.291979,1.28263,1.297494,1.26151,1.141755,1.08044,0.959346,0.865543,0.716534,0.630666,0.572404,0.506499,0.507764,0.473599,0.478199,0.47557,0.48255,0.446497,0.48316,0.485429,0.458921
STD-08,0,InnoSpectra_NIR-S_demo,3,6.1538,0.278206,0.282274,0.247231,0.261876,0.272583,0.279696,0.268699,0.299984,0.258196,0.290724,0.279867,0.285314,0.270031,0.255482,0.278794,0.300053,0.286677,0.290162,0.350098,0.408253,0.466974,0.562706,0.633569,0.650455,0.633885,0.573943,0.478204,0.402543,0.348793,0.383476,0.383704,0.428304,0.457643,0.571047,0.694752,0.857589,0.996522,1.116425,1.180019,1.211499,1.214224,1.195217,1.191808,1.108523,0.99918,0.871328,0.7771,0.639677,0.54451,0.509367,0.452854,0.4348,0.383196,0.389724,0.361275,0.370839,0.384417,0.37641,0.363209,0.356443
STD-09,0,InnoSpectra_NIR-S_demo,1,6.9231,0.243425,0.228637,0.267715,0.283376,0.2693,0.299789,0.233934,0.227595,0.256348,0.231022,0.265407,0.280021,0.280447,0.28604,0.268914,0.275997,0.290798,0.305482,0.3599,0.398909,0.488505,0.566962,0.663567,0.671711,0.639844,0.557354,0.451572,0.403707,0.331446,0.352159,0.342266,0.417693,0.446344,0.556176,0.693727,0.852997,0.997464,1.077298,1.153797,1.150066,1.211144,1.158346,1.127437,1.062189,0.960184,0.870047,0.736617,0.627065,0.547366,0.508461,0.447166,0.368165,0.377574,0.353807,0.363734,0.371851,0.361251,0.372343,0.392229,0.336125
STD-09,0,InnoSpectra_NIR-S_demo,2,6.9231,0.252727,0.250186,0.267577,0.255082,0.266212,0.236017,0.268974,0.248645,0.282302,0.305751,0.278437,0.290179,0.29893,0.313314,0.30753,0.321197,0.302273,0.331131,0.346774,0.471793,0.499234,0.625523,0.682666,0.705621,0.653076,0.601581,0.521054,0.457643,0.403103,0.399371,0.390261,0.461212,0.492173,0.665614,0.745496,0.911471,1.078123,1.170485,1.237432,1.260729,1.238613,1.227327,1.184808,1.131821,1.041901,0.925415,0.8006,0.702543,0.628527,0.572807,0.503969,0.493087,0.448448,0.447128,0.458368,0.423815,0.410445,0.423121,0.442305,0.430466
STD-09,0,InnoSpectra_NIR-S_demo,3,6.9231,0.204777,0.237796,0.234683,0.206739,0.245636,0.248537,0.255126,0.246765,0.252455,0.24312,0.242354,0.260946,0.278554,0.261301,0.304425,0.276751,0.301858,0.296072,0.385309,0.4356,0.531314,0.581116,0.697058,0.705661,0.657821,0.592087,0.492097,0.44679,0.424286,0.382386,0.415753,0.424449,0.463334,0.607954,0.783318,0.951397,1.07345,1.205025,1.248699,1.259607,1.257396,1.270679,1.212356,1.134707,1.061804,0.922294,0.835483,0.749191,0.628988,0.579088,0.539179,0.470083,0.473549,0.463388,0.460273,0.454924,0.49386,0.460193,0.477834,0.494092
STD-10,0,InnoSpectra_NIR-S_demo,1,7.6923,0.190052,0.205035,0.228296,0.212413,0.219494,0.245229,0.216082,0.226698,0.259633,0.24167,0.220337,0.22196,0.254251,0.218433,0.225952,0.259835,0.279539,0.307449,0.30478,0.399508,0.428192,0.570903,0.71229,0.71408,0.663517,0.554435,0.453987,0.414469,0.34327,0.313664,0.348031,0.379923,0.418618,0.526852,0.723435,0.847062,1.036009,1.09987,1.133868,1.142102,1.160413,1.148274,1.118483,1.018854,0.939447,0.811462,0.697033,0.618724,0.530451,0.430704,0.411129,0.345103,0.337872,0.372069,0.361283,0.33357,0.347065,0.359531,0.369051,0.370896
STD-10,0,InnoSpectra_NIR-S_demo,2,7.6923,0.217589,0.177876,0.20956,0.232402,0.225309,0.222061,0.215718,0.229181,0.241901,0.240319,0.268406,0.266274,0.233951,0.275335,0.264188,0.282988,0.270558,0.320295,0.336375,0.436254,0.513392,0.611264,0.67145,0.692622,0.699575,0.585704,0.490983,0.43908,0.384055,0.376178,0.353314,0.373253,0.457208,0.594453,0.729522,0.916536,1.074073,1.162234,1.19999,1.206161,1.217105,1.206173,1.148031,1.095919,0.987873,0.876118,0.764452,0.68166,0.580244,0.537781,0.453712,0.449307,0.394388,0.411171,0.397272,0.397133,0.388485,0.407923,0.42044,0.383282
STD-10,0,InnoSpectra_NIR-S_demo,3,7.6923,0.249322,0.248594,0.227824,0.266477,0.238485,0.265109,0.263522,0.253976,0.251985,0.267266,0.298731,0.288818,0.271819,0.297674,0.317974,0.332852,0.279187,0.344019,0.389442,0.432322,0.509565,0.66446,0.713619,0.720376,0.694732,0.645257,0.576254,0.476627,0.397844,0.416684,0.372928,0.45541,0.517742,0.637567,0.776559,0.958487,1.097458,1.184215,1.228362,1.273211,1.234878,1.231059,1.182754,1.128881,1.027946,0.944173,0.836327,0.718439,0.630558,0.548191,0.478492,0.475618,0.454706,0.424693,0.443312,0.455421,0.425114,0.457741,0.452766,0.43988
STD-11,0,InnoSpectra_NIR-S_demo,1,8.4615,0.291111,0.317839,0.308184,0.311074,0.296637,0.320449,0.337253,0.298929,0.290114,0.307796,0.322483,0.310852,0.302416,0.362844,0.331831,0.331525,0.308717,0.398258,0.403185,0.528461,0.618729,0.723238,0.800658,0.848766,0.783952,0.712121,0.594453,0.511153,0.489296,0.46087,0.433521,0.467916,0.547332,0.651956,0.820395,0.990337,1.137862,1.26113,1.273974,1.295213,1.267427,1.29421,1.23106,1.157688,1.062339,0.918967,0.84823,0.727334,0.633302,0.594412,0.570723,0.497963,0.517665,0.442712,0.481373,0.464922,0.480581,0.515278,0.492836,0.458586
STD-11,0,InnoSpectra_NIR-S_demo,2,8.4615,0.339767,0.336425,0.356736,0.354999,0.387575,0.367851,0.376338,0.364192,0.388705,0.353829,0.349361,0.392115,0.392278,0.382924,0.42431,0.375774,0.427341,0.453318,0.451782,0.55826,0.664532,0.758375,0.842533,0.871038,0.817556,0.720052,0.653709,0.556877,0.498655,0.485782,0.469641,0.504808,0.593022,0.685856,0.851224,1.045554,1.21506,1.25981,1.295587,1.285648,1.296982,1.302293,1.241495,1.214355,1.07124,0.937296,0.841928,0.792525,0.656861,0.605333,0.54501,0.508295,0.527405,0.475832,0.460147,0.476358,0.478056,0.498379,0.494755,0.516331
STD-11,0,InnoSpectra_NIR-S_demo,3,8.4615,0.238287,0.251104,0.216352,0.235414,0.198016,0.235054,0.231559,0.241291,0.286714,0.281476,0.241717,0.3023,0.297252,0.2825,0.282363,0.288777,0.303377,0.340707,0.372289,0.470873,0.551759,0.656574,0.751289,0.787933,0.751301,0.656469,0.528482,0.445364,0.385571,0.390144,0.41478,0.437345,0.500562,0.601239,0.761912,0.941745,1.130883,1.183287,1.220713,1.250326,1.23578,1.239327,1.178696,1.121357,1.021128,0.886014,0.808222,0.720934,0.604881,0.549753,0.516812,0.501675,0.463158,0.48705,0.436462,0.475771,0.431862,0.446899,0.435102,0.471101
STD-12,0,InnoSpectra_NIR-S_demo,1,9.2308,0.272293,0.231646,0.209275,0.223455,0.231975,0.267899,0.249607,0.243925,0.228053,0.256688,0.216471,0.266634,0.265214,0.241988,0.226535,0.246174,0.265687,0.326835,0.370499,0.411537,0.519802,0.627842,0.747867,0.760331,0.736721,0.627211,0.501671,0.396405,0.323894,0.334733,0.306113,0.371182,0.393628,0.487093,0.692149,0.87136,1.057162,1.127457,1.147067,1.158865,1.125413,1.128847,1.115279,1.002532,0.884111,0.788005,0.66373,0.570328,0.519238,0.407591,0.35496,0.351676,0.315013,0.306673,0.29228,0.303903,0.323717,0.309613,0.292271,0.294265
STD-12,0,InnoSpectra_NIR-S_demo,2,9.2308,0.246595,0.242445,0.243854,0.247122,0.23066,0.24378,0.255431,0.276498,0.256346,0.25585,0.243911,0.27743,0.258453,0.269926,0.276015,0.262346,0.293077,0.33368,0.350049,0.438795,0.545596,0.681045,0.80657,0.806455,0.778634,0.664601,0.5398,0.451288,0.394188,0.362213,0.369746,0.404451,0.456569,0.603049,0.747776,0.929494,1.083471,1.186873,1.189959,1.179305,1.188061,1.198256,1.130872,1.063902,0.94738,0.862581,0.73499,0.634763,0.547647,0.493464,0.431006,0.410397,0.394543,0.37079,0.348981,0.387583,0.35968,0.349351,0.384483,0.385995
STD-12,0,InnoSpectra_NIR-S_demo,3,9.2308,0.234181,0.266193,0.284936,0.262264,0.308651,0.29981,0.287203,0.294229,0.312966,0.347738,0.349117,0.297157,0.336094,0.305964,0.334096,0.327693,0.353997,0.379797,0.445645,0.511323,0.649445,0.757512,0.837812,0.909402,0.849056,0.744696,0.638075,0.554432,0.481384,0.450625,0.465509,0.492815,0.559164,0.707397,0.858387,1.060679,1.193218,1.261509,1.308989,1.323473,1.302245,1.298483,1.238237,1.191303,1.080181,0.979788,0.883196,0.761636,0.674319,0.615461,0.593839,0.548918,0.554837,0.536985,0.543472,0.528212,0.547737,0.543402,0.502691,0.563413
STD-13,0,InnoSpectra_NIR-S_demo,1,10.0,0.231891,0.267957,0.253495,0.250725,0.256045,0.247524,0.293291,0.265064,0.281793,0.287169,0.293122,0.286584,0.305608,0.284248,0.333788,0.277302,0.328899,0.382158,0.382926,0.511822,0.642544,0.729356,0.861571,0.85768,0.825152,0.710857,0.574921,0.468531,0.410542,0.366649,0.398558,0.429125,0.510348,0.611494,0.770103,0.964647,1.176148,1.192478,1.211162,1.261617,1.231123,1.196947,1.159458,1.094016,0.955256,0.865646,0.77386,0.673579,0.597807,0.518336,0.461316,0.425642,0.426869,0.405064,0.408138,0.42208,0.419511,0.378551,0.421528,0.387631
STD-13,0,InnoSpectra_NIR-S_demo,2,10.0,0.125124,0.115553,0.12397,0.15179,0.175872,0.169516,0.148765,0.138287,0.161289,0.185845,0.182516,0.156966,0.186812,0.200772,0.181118,0.220969,0.246909,0.268595,0.295551,0.396888,0.523584,0.650797,0.772916,0.794382,0.748585,0.65158,0.518273,0.401747,0.362521,0.329571,0.352997,0.379938,0.439363,0.577835,0.730543,0.909351,1.124511,1.205044,1.202979,1.169711,1.171566,1.188639,1.130359,1.055453,0.975004,0.851287,0.746225,0.678724,0.586722,0.528985,0.48222,0.461333,0.441138,0.405637,0.413419,0.445581,0.424002,0.454484,0.425055,0.411992
STD-13,0,InnoSpectra_NIR-S_demo,3,10.0,0.260353,0.262451,0.243961,0.290156,0.28158,0.271627,0.262925,0.304269,0.296631,0.310635,0.308664,0.318021,0.307921,0.287308,0.318753,0.337831,0.312841,0.357792,0.444108,0.488464,0.652629,0.744014,0.831438,0.88236,0.834007,0.755162,0.581593,0.497056,0.430974,0.400884,0.424465,0.445503,0.519933,0.629098,0.820472,0.989347,1.16572,1.264402,1.274491,1.25678,1.260404,1.235421,1.193124,1.114516,1.033529,0.926595,0.790472,0.711066,0.626423,0.566602,0.497084,0.458402,0.456158,0.467558,0.427843,0.460052,0.445057,0.449671,0.452488,0.44639
VINEGAR_WHITE_W4REF,0,InnoSpectra_NIR-S_demo,1,4.5,0.247632,0.256626,0.240208,0.256861,0.281516,0.287073,0.26882,0.277333,0.275774,0.273488,0.300988,0.290258,0.327262,0.270489,0.281937,0.296819,0.3282,0.340741,0.35826,0.361708,0.475693,0.507117,0.605851,0.602798,0.570868,0.535742,0.51658,0.443035,0.407728,0.382212,0.44016,0.467249,0.560434,0.636978,0.781938,0.908907,1.045799,1.184737,1.237193,1.273414,1.308972,1.324321,1.257762,1.227853,1.104459,0.963675,0.853294,0.768765,0.678963,0.612791,0.569952,0.546029,0.519203,0.505204,0.479125,0.538904,0.515972,0.527072,0.524937,0.547946
VINEGAR_WHITE_W4REF,0,InnoSpectra_NIR-S_demo,2,4.5,0.209987,0.220015,0.237454,0.242842,0.257492,0.232216,0.254196,0.242036,0.231418,0.276308,0.263051,0.252173,0.26049,0.286596,0.282755,0.287034,0.309306,0.313641,0.311636,0.399057,0.434396,0.497657,0.511239,0.514657,0.552417,0.500438,0.457428,0.421037,0.409047,0.41461,0.430149,0.429909,0.494142,0.602276,0.753101,0.900201,1.016286,1.113351,1.209069,1.246536,1.285646,1.284089,1.237027,1.142939,1.037941,0.941241,0.841464,0.730145,0.657701,0.573163,0.551198,0.507645,0.478025,0.461628,0.467929,0.471756,0.499033,0.477565,0.491179,0.449773
VINEGAR_WHITE_W4REF,0,InnoSpectra_NIR-S_demo,3,4.5,0.284269,0.264989,0.292143,0.33692,0.326955,0.329744,0.311693,0.333391,0.336736,0.335381,0.317502,0.32623,0.334596,0.394075,0.313904,0.37979,0.383727,0.378883,0.415419,0.449955,0.509498,0.550488,0.573871,0.668908,0.634383,0.536831,0.527582,0.459367,0.450552,0.422379,0.438094,0.523208,0.54358,0.641657,0.782885,0.946795,1.056169,1.182001,1.25417,1.275579,1.299288,1.314656,1.261985,1.215641,1.058111,0.983682,0.858699,0.734439,0.669573,0.580306,0.550426,0.542315,0.492389,0.503871,0.493933,0.479504,0.484232,0.494777,0.499473,0.488677
VINEGAR_B_DEMO,0,InnoSpectra_NIR-S_demo,1,6.2,0.29322,0.320939,0.326028,0.281813,0.325025,0.349147,0.34231,0.3248,0.339765,0.308326,0.349136,0.346,0.358191,0.343729,0.360793,0.347109,0.381862,0.38976,0.386425,0.444644,0.55709,0.645535,0.712662,0.703368,0.68751,0.61492,0.534599,0.479739,0.46734,0.427668,0.477284,0.458949,0.536125,0.648409,0.788124,0.946194,1.064591,1.187831,1.268554,1.282816,1.292982,1.281687,1.231047,1.157063,1.047852,0.967985,0.839573,0.736014,0.646134,0.572395,0.5037,0.495246,0.463968,0.454691,0.462032,0.4791,0.439865,0.458787,0.487714,0.442005
VINEGAR_B_DEMO,0,InnoSpectra_NIR-S_demo,2,6.2,0.300933,0.31819,0.293331,0.314189,0.305978,0.335987,0.249114,0.294556,0.311007,0.317705,0.347288,0.341235,0.31887,0.304088,0.351845,0.34213,0.342736,0.380776,0.431528,0.46366,0.557893,0.584815,0.692585,0.73902,0.692457,0.598452,0.515471,0.479507,0.421871,0.416642,0.408152,0.455153,0.542668,0.618315,0.75747,0.914072,1.083237,1.191744,1.240285,1.217924,1.278836,1.253503,1.22639,1.13508,1.073381,0.97429,0.845259,0.757222,0.646173,0.571116,0.49331,0.484949,0.468727,0.492003,0.444366,0.46054,0.470989,0.463084,0.464356,0.485355
VINEGAR_B_DEMO,0,InnoSpectra_NIR-S_demo,3,6.2,0.212813,0.239569,0.23913,0.228943,0.237262,0.190505,0.234665,0.223987,0.210423,0.250978,0.236198,0.234849,0.22045,0.256269,0.278443,0.266092,0.285876,0.309581,0.327807,0.378517,0.450893,0.538115,0.596921,0.628992,0.592458,0.556436,0.449314,0.430403,0.358935,0.348513,0.380514,0.391526,0.433638,0.578298,0.729197,0.837671,1.028926,1.111127,1.170056,1.228356,1.191952,1.197378,1.186513,1.095322,0.999885,0.869099,0.75489,0.653591,0.58531,0.508116,0.475328,0.480991,0.387166,0.411303,0.400651,0.383337,0.405686,0.389769,0.384782,0.411011
"""

In [ ]:
df = load_spectra()
meta, X_all, wl = split_spectra(df)
print(f"讀入 {len(df)} 筆光譜，共 {meta['sample_id'].nunique()} 個樣品、"
      f"{len(wl)} 個波長點（{wl.min():.0f}-{wl.max():.0f} nm）。")
df.head()

### （選用）在 Colab 上傳班級實際資料

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_spectra(fname)
#     meta, X_all, wl = split_spectra(df)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_spectra('本機路徑或URL')")

## 2. 資料總覽 (Data Overview)

先畫幾條代表性的光譜，確認資料讀取正確：可以看到隨著醋酸濃度升高，1450 nm 附近的水 O–H 大帶形狀被逐漸疊加的醋酸訊號微幅改變。

In [ ]:
show_idx = [0, 1, 2, 3, 4, 5]  # a few standards spanning low -> high concentration
example_labels = [f"{meta.loc[i, 'sample_id']} ({meta.loc[i, 'acidity_pct']:.2f}%)" for i in show_idx]
ax = plot_spectra(wl, X_all[show_idx], labels=example_labels)
ax.set_title("Example raw NIR spectra (increasing acetic acid %)")
plt.tight_layout()
plt.show()

## 3. 校正集／測試集切分 (Calibration / Test Split)

**測試集只能用一次**，用來在最後誠實地評估模型；它絕對不能參與挑選前處理方法或潛在變數（LV）數，否則誤差估計會系統性偏樂觀（過度樂觀，optimistic bias）。

切分要在**樣品層級**（sample-level）進行，不是在光譜（重複）層級：同一個樣品的 3 次重複必須整個一起分到校正集或測試集，不能拆開——這正是後面第 4 節交叉驗證要用 `GroupKFold`/`LeaveOneGroupOut` 分組的同一個理由。

### Kennard–Stone 演算法

Kennard–Stone (KS) 是化學計量學裡常用的「代表性子集挑選」演算法，操作在 **X 空間**（這裡是光譜空間）：

1. 先挑出彼此歐氏距離（Euclidean distance）最遠的一對樣品（保證涵蓋光譜空間的兩個極端）。
2. 之後每一輪，從剩下的樣品中挑出「離目前已選集合的最小距離」最大的那一個（也就是離已選集合最遠的樣品），逐步把樣品空間填滿、涵蓋範圍。
3. 重複直到達到想要的校正集大小。

因為第一步保證挑出最極端的一對，KS 選出的校正集自然會涵蓋樣品範圍的兩端，不容易讓測試集需要「外插」（extrapolation）。

> **另一種常見做法：依濃度分層抽樣（stratified by concentration）**——當光譜只受單一分析物濃度支配（像本週的醋酸水溶液）時，KS 在 X 空間選出的順序通常會接近直接按濃度排序後分層抽樣的結果；但當光譜同時受多個成分或基質效應影響時，KS 在 X 空間工作能捕捉到單看濃度看不出來的差異，是更一般化的做法。本筆記本採用 KS，並在下面印出結果讓你對照兩者的差異。

In [ ]:
def kennard_stone(X, n_select):
    """Kennard-Stone sample-selection algorithm.

    Greedily builds a subset of `n_select` rows of X that is spread out
    across the multivariate X-space: start from the most distant pair, then
    repeatedly add whichever remaining row has the largest *minimum*
    distance to the already-selected set. Returns the selected row indices
    in the order they were picked (index 0 and 1 are the initial extreme
    pair).
    """
    X = np.asarray(X, dtype=float)
    n = X.shape[0]
    D = squareform(pdist(X))
    i0, i1 = np.unravel_index(np.argmax(D), D.shape)
    selected = [int(i0), int(i1)]
    remaining = [i for i in range(n) if i not in selected]
    while len(selected) < n_select:
        min_d = D[np.ix_(remaining, selected)].min(axis=1)
        nxt = remaining[int(np.argmax(min_d))]
        selected.append(nxt)
        remaining.remove(nxt)
    return selected


# Run KS on the per-sample MEAN spectrum (one row per unique sample_id), so
# replicates are never split between calibration and test.
sample_table = (
    meta.assign(row=np.arange(len(meta)))
    .groupby("sample_id", sort=False)
    .agg(acidity_pct=("acidity_pct", "first"), rows=("row", list))
    .reset_index()
)
mean_spectra = np.array([X_all[r["rows"]].mean(axis=0) for _, r in sample_table.iterrows()])

N_UNIQUE = len(sample_table)
N_CAL_SAMPLES = int(round(0.72 * N_UNIQUE))  # ~72% of samples -> calibration
cal_sample_pos = kennard_stone(mean_spectra, N_CAL_SAMPLES)
test_sample_pos = [i for i in range(N_UNIQUE) if i not in cal_sample_pos]

cal_sample_ids = set(sample_table.loc[cal_sample_pos, "sample_id"])
test_sample_ids = set(sample_table.loc[test_sample_pos, "sample_id"])

cal_mask = meta["sample_id"].isin(cal_sample_ids).to_numpy()
test_mask = meta["sample_id"].isin(test_sample_ids).to_numpy()

X_cal, y_cal, g_cal = X_all[cal_mask], meta.loc[cal_mask, "acidity_pct"].to_numpy(), meta.loc[cal_mask, "sample_id"].to_numpy()
X_test, y_test, g_test = X_all[test_mask], meta.loc[test_mask, "acidity_pct"].to_numpy(), meta.loc[test_mask, "sample_id"].to_numpy()

print(f"樣品總數 n={N_UNIQUE}；校正集樣品數={len(cal_sample_ids)}，測試集樣品數={len(test_sample_ids)}")
print(f"校正集光譜數={cal_mask.sum()}，測試集光譜數={test_mask.sum()}（測試集佔全部光譜 {test_mask.mean()*100:.1f}%）")
print("校正集樣品：", sorted(cal_sample_ids))
print("測試集樣品：", sorted(test_sample_ids))

assert 0.20 <= test_mask.mean() <= 0.35, "測試集比例應落在 20-35% 之間"
assert sample_table["acidity_pct"].min() in meta.loc[cal_mask, "acidity_pct"].values, "校正集應涵蓋濃度下限，避免測試集外插"
assert sample_table["acidity_pct"].max() in meta.loc[cal_mask, "acidity_pct"].values, "校正集應涵蓋濃度上限，避免測試集外插"
print("檢查通過：校正集涵蓋了濃度的兩個極端，測試集比例落在合理範圍。")

## 4. 前處理比較 (Preprocessing Comparison)

比較三種前處理方法對 **RMSECV**（交叉驗證均方根誤差）的影響：

- **raw**：不做前處理（PLS 內部仍會自動置中）。
- **SNV**（Standard Normal Variate）：每條光譜各自減去自己的平均值、除以自己的標準差，扣掉整體散射造成的乘性/加性效應。
- **SG 一階導數**（Savitzky–Golay 1st derivative）：用局部多項式擬合再微分，扣掉平緩的基線飄移，只保留形狀變化較快的訊號。

**關鍵規則：交叉驗證一定要用 `GroupKFold` 或 `LeaveOneGroupOut`，以 `sample_id` 分組**——如果直接對所有光譜（含重複）做普通的 `KFold`，同一樣品的其中 2 次重複可能進訓練折、第 3 次重複進驗證折，驗證折等於已經看過幾乎一樣的光譜，RMSECV 會系統性被低估（過度樂觀）。第 8 節的練習會實際示範這個資料外洩（data leakage）現象。

In [ ]:
def snv(X):
    m = X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True)
    return (X - m) / s


def sg_deriv1(X, window=9, polyorder=2):
    return savgol_filter(X, window_length=window, polyorder=polyorder, deriv=1, axis=1)


PREP = {
    "raw": (X_cal, X_test),
    "SNV": (snv(X_cal), snv(X_test)),
    "SG_1st_deriv": (sg_deriv1(X_cal), sg_deriv1(X_test)),
}

MAX_LV = 15


def rmsecv_curve_grouped(Xc, yc, groups, max_lv):
    """RMSEC and RMSECV (LeaveOneGroupOut by `groups`) for LV = 1..max_lv."""
    logo = LeaveOneGroupOut()
    out = []
    for nlv in range(1, max_lv + 1):
        pred = np.zeros_like(yc, dtype=float)
        for tr, va in logo.split(Xc, yc, groups):
            m = PLSRegression(n_components=nlv)
            m.fit(Xc[tr], yc[tr])
            pred[va] = m.predict(Xc[va]).ravel()
        rmsecv = float(np.sqrt(np.mean((pred - yc) ** 2)))
        m_full = PLSRegression(n_components=nlv).fit(Xc, yc)
        rmsec = float(np.sqrt(np.mean((m_full.predict(Xc).ravel() - yc) ** 2)))
        out.append((nlv, rmsec, rmsecv))
    return out


results_by_prep = {name: rmsecv_curve_grouped(Xc, y_cal, g_cal, MAX_LV) for name, (Xc, Xt) in PREP.items()}

print("== 各前處理法 RMSECV 最小值 ==")
best_prep, best_lv, best_rmsecv = None, None, np.inf
for name, curve in results_by_prep.items():
    lv_min = min(curve, key=lambda r: r[2])
    print(f"  {name:<14s}: 最小 RMSECV={lv_min[2]:.4f}  (LV={lv_min[0]})")
    if lv_min[2] < best_rmsecv:
        best_prep, best_lv, best_rmsecv = name, lv_min[0], lv_min[2]
print(f"選用前處理：{best_prep}，最佳 LV={best_lv}，RMSECV={best_rmsecv:.4f}")

## 5. RMSEC / RMSECV vs LV 數

用選定的前處理法，畫出 RMSEC（校正集配適誤差，只會隨 LV 增加持續下降）與 RMSECV（分組交叉驗證誤差，會在某個 LV 附近出現最小值）的曲線。RMSEC 與 RMSECV 分開拉大的地方，就是模型開始過擬合（overfitting）的訊號。

In [ ]:
Xc_best, Xt_best = PREP[best_prep]
curve = results_by_prep[best_prep]
lv_list = [r[0] for r in curve]
rmsec_list = [r[1] for r in curve]
rmsecv_list = [r[2] for r in curve]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(lv_list, rmsec_list, "o-", color="tab:blue", label="RMSEC (calibration fit)")
ax.plot(lv_list, rmsecv_list, "s-", color="tab:orange", label="RMSECV (LeaveOneGroupOut CV)")
ax.axvline(best_lv, color="grey", linestyle="--", linewidth=1, label=f"chosen LV = {best_lv}")
ax.set_xlabel("Number of latent variables (LV)")
ax.set_ylabel("RMSE (% w/v)")
ax.set_title(f"RMSEC vs RMSECV  (preprocessing = {best_prep})")
ax.legend()
plt.tight_layout()
plt.show()

## 6. 最終模型與品質指標 (Final Model & Figures of Merit)

用選定的前處理法與 LV 數，在**整個校正集**上重新配適一次最終模型，然後**只用一次**測試集算出：

- **RMSEP**（測試集均方根誤差）
- **R²**（測試集預測值與參考值的相關程度）
- **bias**（測試集預測值的平均系統性偏差）
- **RPD** = SD(y_test) / RMSEP（相對於測試集本身變異的表現，經驗法則 RPD > 3 大致可用於篩選等級）

因為 y（`acidity_pct`）本身帶有約 `PREP_SD` = 0.05% w/v 的滴定參考值不確定度，**RMSEP 理論上很難顯著小於這個數字**——這是參考方法精密度設下的天花板。

In [ ]:
final_model = PLSRegression(n_components=best_lv).fit(Xc_best, y_cal)
pred_test = final_model.predict(Xt_best).ravel()

RMSEP = float(np.sqrt(np.mean((pred_test - y_test) ** 2)))
resid_test = pred_test - y_test
bias_test = float(np.mean(resid_test))
ss_res = float(np.sum((y_test - pred_test) ** 2))
ss_tot = float(np.sum((y_test - y_test.mean()) ** 2))
R2_test = 1 - ss_res / ss_tot
SD_ytest = float(np.std(y_test, ddof=1))
RPD = SD_ytest / RMSEP

RMSEC_final = float(np.sqrt(np.mean((final_model.predict(Xc_best).ravel() - y_cal) ** 2)))
RMSECV_final = best_rmsecv

print(f"前處理={best_prep}  LV={best_lv}")
print(f"RMSEC={RMSEC_final:.4f}  RMSECV={RMSECV_final:.4f}  RMSEP={RMSEP:.4f}  "
      f"R2(test)={R2_test:.4f}  bias(test)={bias_test:+.4f}  RPD={RPD:.2f}")
print(f"參考值不確定度 PREP_SD={PREP_SD:.2f}%  ->  RMSEP 理論下限大約在這個量級附近")

In [ ]:
axis_lo = min(y_cal.min(), y_test.min(), pred_test.min()) - 0.5
axis_hi = max(y_cal.max(), y_test.max(), pred_test.max()) + 0.5

# Calibration-side points are shown using LeaveOneGroupOut CV predictions at
# the chosen LV, to avoid the optimistic bias of "fit and predict on the same
# calibration data".
logo = LeaveOneGroupOut()
cv_pred_cal = np.zeros_like(y_cal, dtype=float)
for tr, va in logo.split(Xc_best, y_cal, g_cal):
    m = PLSRegression(n_components=best_lv).fit(Xc_best[tr], y_cal[tr])
    cv_pred_cal[va] = m.predict(Xc_best[va]).ravel()

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(y_cal, cv_pred_cal, s=45, color="tab:blue", alpha=0.8, label="Calibration (CV predicted)")
ax.scatter(y_test, pred_test, s=70, color="tab:orange", marker="^", label="Test (independent)")
ax.plot([axis_lo, axis_hi], [axis_lo, axis_hi], "--", color="grey", linewidth=1.5, label="1:1 line")
ax.set_xlim(axis_lo, axis_hi)
ax.set_ylim(axis_lo, axis_hi)
ax.set_xlabel("Reference acidity (%w/v, W4 titration)")
ax.set_ylabel("PLS-predicted acidity (%w/v)")
ax.set_title("Predicted vs reference")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

## 7. Model Report Card

把這個模型的所有關鍵資訊整理成一張表，方便之後（W14／W17）快速對照與比較。

In [ ]:
report_card = pd.DataFrame([{
    "instrument": meta.loc[cal_mask, "instrument"].iloc[0],
    "preprocessing": best_prep,
    "n_LV": best_lv,
    "n_calibration_spectra": int(cal_mask.sum()),
    "n_test_spectra": int(test_mask.sum()),
    "RMSEC_pct": round(RMSEC_final, 4),
    "RMSECV_pct": round(RMSECV_final, 4),
    "RMSEP_pct": round(RMSEP, 4),
    "R2_test": round(R2_test, 4),
    "bias_pct": round(bias_test, 4),
    "RPD": round(RPD, 2),
    "applicable_range_pct": f"{y_cal.min():.1f}-{y_cal.max():.1f}",
}])
report_card

## 8. 其他工具的等效寫法 (R / Orange)

### R（`pls` 套件）

```r
library(pls)
mod <- plsr(y ~ X, ncomp = 15, data = cal_df, validation = "CV",
            segments = cal_df$sample_id)   # 分組交叉驗證，等同 GroupKFold
plot(RMSEP(mod))                            # 找 RMSECV 最小值對應的 ncomp
pred <- predict(mod, ncomp = best_lv, newdata = test_df)
```

`validation="CV"` 搭配 `segments=` 指定分組向量，效果等同本筆記本用 `LeaveOneGroupOut`／`GroupKFold` 以 `sample_id` 分組交叉驗證，避免同一樣品的重複同時出現在訓練與驗證。

### Orange（Test and Score）

在 Orange 中，`File`（讀入光譜＋參考值）→ `Preprocess`（SNV／一階導數等）→ `Test and Score`（PLS learner，指定交叉驗證的分組欄位為 `sample_id`）→ `Predictions`（套用固定模型到測試集）。邏輯與本筆記本完全一致：**配適 → 分組交叉驗證選 LV → 用固定模型預測測試集**，只是介面不同。

## 9. 載入真實 InnoSpectra／Avantes 匯出資料

只要把班上實際量測的資料整理成本筆記本開頭定義的共用長格式（`sample_id, group, instrument, rep, acidity_pct, <波長...>`），存成 CSV，就能直接用 `load_spectra()` 載入、套用第 3–7 節完全一樣的流程。若你的資料是用 SpectraView 匯出（`layout="rows"`），`load_spectra()` 也會自動判斷並轉換格式。

In [ ]:
# 範例（示範用法，預設不會執行，因為示範路徑不存在）：
#
# real_df = load_spectra("data/w13_class_nir_export.csv")
# real_meta, real_X, real_wl = split_spectra(real_df)
# plot_spectra(real_wl, real_X[:5], labels=real_meta["sample_id"].iloc[:5].tolist())
#
# 或直接餵一個已發布的 Google 試算表 CSV 連結／SpectraView 匯出檔路徑：
# real_df = load_spectra("https://.../pub?output=csv")

print("此 cell 僅示範用法（見上方註解），因為示範環境沒有實際的班級匯出檔可供載入。")
print("實際使用時，把 load_spectra() 的路徑換成你們班的匯出 CSV 即可套用同一套流程。")

## 10. 練習題 (Exercises)

請在每題下方的空白程式碼區塊中作答（可以用文字 print 說明，也可以搭配計算/繪圖）。

### 問題 1

比較三種前處理法在最佳 LV 時的 RMSECV（見 `results_by_prep`），哪一種對這批模擬資料最好？試著解釋為什麼（提示：這批資料的「雜訊」主要是哪一種型態：整體散射／基線飄移，還是隨機雜訊？）。

In [ ]:
# TODO: 請在這裡作答

### 問題 2（資料外洩示範）

把第 4 節的分組交叉驗證（`LeaveOneGroupOut`，以 `sample_id` 分組）換成**不分組**的普通 `KFold`（例如 5 折），對校正集重新算一次 RMSECV（固定用 `best_prep` 與 `best_lv`）。比較兩者的數字，並解釋為什麼不分組會讓 RMSECV 看起來比較樂觀。

In [ ]:
# TODO: 請在這裡作答

### 問題 3

把第 3 節的 Kennard–Stone 切分，改成單純依 `acidity_pct` 由小到大排序後的分層抽樣（例如每 4 個樣品取 1 個當測試集）。比較新切分下的測試集組成（涵蓋的濃度範圍）與最終 RMSEP，跟原本 KS 切分的結果有什麼不同？

In [ ]:
# TODO: 請在這裡作答

### 問題 4

本模型的 RPD 大約是多少？對照課堂講的「RPD > 3 大致可用於篩選（screening）等級」的經驗法則，這個模型適合拿來做哪一種等級的應用？如果 RPD 不到 3，可能可以怎麼改善（前處理？增加校正集樣品數？換更寬的濃度範圍？）？

In [ ]:
# TODO: 請在這裡作答

### 問題 5

`PREP_SD`（W4 滴定參考值不確定度）設定為 0.05% w/v。如果把它改成 0.20%（重新產生資料、重跑整個流程），你預期 RMSEP 會怎麼變化？為什麼參考方法本身的精密度會限制模型 RMSEP 的下限？

In [ ]:
# TODO: 請在這裡作答

## 11. 匯出結果 (Export)

把 model report card 存成 CSV。若在 Colab 環境執行，會另外觸發瀏覽器下載；在本機執行則只會存檔。

In [ ]:
output_path = "nb11_model_report_card.csv"
report_card.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")